# Topic 1 — Linear Regression

> **Goal of this notebook:** understand Linear Regression deeply — the intuition, the math behind it, its assumptions — and then apply it end-to-end on a real dataset to predict house prices.

**Contents**
1. Concept & intuition
2. The mathematics (cost function, gradient descent, normal equation)
3. Assumptions of linear regression
4. The dataset: California Housing
5. Exploratory data analysis
6. Training a model with scikit-learn
7. Evaluating the model
8. Building it from scratch (gradient descent)
9. Pros, cons & when to use
10. Summary

## 1. Concept & Intuition

**Linear Regression** is the simplest supervised learning algorithm for **regression** (predicting a continuous number, e.g. price, temperature, salary).

The core idea: assume the target `y` is a **straight-line (linear) combination** of the input features `x`. With one feature this is the familiar line:

$$\hat{y} = w x + b$$

where `w` is the slope (weight) and `b` is the intercept (bias). With many features it becomes:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

**Intuition:** we are drawing the best-fitting line (or hyperplane) through the data. "Best" means the line that makes our predictions as close as possible to the true values. The algorithm's job is to *learn* the weights `w` and bias `b` that minimise the prediction error.

## 2. The Mathematics

### Cost function — Mean Squared Error (MSE)

We measure how wrong the model is using the **Mean Squared Error**:

$$J(w, b) = \frac{1}{m} \sum_{i=1}^{m} \left( \hat{y}^{(i)} - y^{(i)} \right)^2$$

where `m` is the number of training examples. We square the errors so positive and negative errors don't cancel, and to penalise large mistakes more heavily. Training = finding `w, b` that **minimise** `J`.

### Method A — Gradient Descent

We iteratively nudge the parameters in the direction that reduces the cost. The gradients are:

$$\frac{\partial J}{\partial w_j} = \frac{2}{m} \sum_{i=1}^{m} \left( \hat{y}^{(i)} - y^{(i)} \right) x_j^{(i)} \qquad \frac{\partial J}{\partial b} = \frac{2}{m} \sum_{i=1}^{m} \left( \hat{y}^{(i)} - y^{(i)} \right)$$

Then update with a learning rate `α`:

$$w_j := w_j - \alpha \frac{\partial J}{\partial w_j} \qquad b := b - \alpha \frac{\partial J}{\partial b}$$

### Method B — The Normal Equation (closed form)

Linear regression also has an exact analytical solution (no iteration needed):

$$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

scikit-learn's `LinearRegression` uses an efficient closed-form/least-squares solver under the hood.

## 3. Assumptions of Linear Regression

Linear regression works best when these hold:

1. **Linearity** — the relationship between features and target is linear.
2. **Independence** — observations are independent of each other.
3. **Homoscedasticity** — the variance of the residuals is constant across all predictions.
4. **Normality of residuals** — errors are roughly normally distributed.
5. **No (or low) multicollinearity** — features are not highly correlated with each other.

We'll visually check a couple of these (linearity & homoscedasticity) using residual plots later.

In [ ]:
# Core libraries used throughout this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: California Housing

We'll use the **California Housing** dataset (built into scikit-learn). Each row is a census block group in California, and the task is to predict the **median house value** (in $100,000s) from features such as median income, average rooms, house age, and location.

This is a classic, well-behaved regression dataset — perfect for showcasing linear regression. (The first run downloads ~400 KB and caches it locally.)

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame

print('Shape:', df.shape)
print('\nFeatures:', list(data.feature_names))
print('Target: MedHouseVal (median house value in $100,000s)')
df.head()

In [ ]:
# Quick statistical summary and a check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print()
df.describe()

## 5. Exploratory Data Analysis

Before modelling, let's understand which features relate to the target. `MedInc` (median income) is usually the strongest predictor of house value.

In [ ]:
# Correlation of each feature with the target
corr = df.corr()['MedHouseVal'].sort_values(ascending=False)
print('Correlation with median house value:\n')
print(corr)

In [ ]:
# Visualise the strongest relationship: median income vs house value
plt.scatter(df['MedInc'], df['MedHouseVal'], alpha=0.15, s=8)
plt.xlabel('Median Income (in $10,000s)')
plt.ylabel('Median House Value (in $100,000s)')
plt.title('Median Income vs House Value')
plt.show()

## 6. Training a Model with scikit-learn

Steps:
1. Split features `X` and target `y`.
2. Split into train / test sets (so we evaluate on unseen data).
3. **Scale** the features — helps interpretability of coefficients and is essential for the from-scratch gradient descent later.
4. Fit `LinearRegression`.

In [ ]:
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Standardise features (mean 0, std 1) — fit on train only to avoid leakage
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_s, y_train)

print('Intercept (b):', round(model.intercept_, 4))
print('\nLearned coefficients (w):')
for name, coef in zip(X.columns, model.coef_):
    print(f'  {name:12s} {coef:+.4f}')

**Reading the coefficients:** because features are standardised, each coefficient tells us the change in house value (in $100,000s) per one-standard-deviation increase in that feature, holding others fixed. `MedInc` should have the largest positive coefficient — confirming income drives prices.

## 7. Evaluating the Model

We judge regression models with:
- **RMSE** (Root Mean Squared Error) — typical error size, in the target's units.
- **MAE** (Mean Absolute Error) — average absolute error, less sensitive to outliers.
- **R²** (coefficient of determination) — fraction of variance explained (1.0 = perfect, 0 = no better than the mean).

In [ ]:
y_pred = model.predict(X_test_s)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'RMSE: {rmse:.4f}  (≈ ${rmse*100000:,.0f})')
print(f'MAE:  {mae:.4f}  (≈ ${mae*100000:,.0f})')
print(f'R²:   {r2:.4f}')

In [ ]:
# Predicted vs actual: points should hug the diagonal
plt.scatter(y_test, y_pred, alpha=0.15, s=8)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual house value')
plt.ylabel('Predicted house value')
plt.title('Predicted vs Actual')
plt.legend()
plt.show()

In [ ]:
# Residual plot: residuals should scatter randomly around 0 (checks homoscedasticity)
residuals = y_test - y_pred
plt.scatter(y_pred, residuals, alpha=0.15, s=8)
plt.axhline(0, color='r', ls='--', lw=2)
plt.xlabel('Predicted value')
plt.ylabel('Residual (actual - predicted)')
plt.title('Residual Plot')
plt.show()

## 8. Building Linear Regression From Scratch

To cement the theory, let's implement gradient descent ourselves and confirm it converges to (almost) the same coefficients as scikit-learn.

In [ ]:
class LinearRegressionGD:
    """Linear Regression trained with batch gradient descent."""

    def __init__(self, lr=0.1, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.w = None
        self.b = 0.0
        self.history = []

    def fit(self, X, y):
        m, n = X.shape
        y = np.asarray(y)
        self.w = np.zeros(n)
        self.b = 0.0
        for _ in range(self.n_iters):
            y_hat = X @ self.w + self.b
            error = y_hat - y
            dw = (2 / m) * (X.T @ error)
            db = (2 / m) * error.sum()
            self.w -= self.lr * dw
            self.b -= self.lr * db
            self.history.append((error ** 2).mean())  # MSE per iteration
        return self

    def predict(self, X):
        return X @ self.w + self.b

In [ ]:
gd = LinearRegressionGD(lr=0.1, n_iters=1000).fit(X_train_s, y_train)

gd_pred = gd.predict(X_test_s)
print('From-scratch R²: ', round(r2_score(y_test, gd_pred), 4))
print('scikit-learn R²:', round(r2, 4))
print('\nIntercept  -> scratch:', round(gd.b, 4), '| sklearn:', round(model.intercept_, 4))

In [ ]:
# Watch the cost drop as gradient descent learns
plt.plot(gd.history)
plt.xlabel('Iteration')
plt.ylabel('MSE (training)')
plt.title('Gradient Descent Convergence')
plt.show()

## 9. Pros, Cons & When to Use

**Pros**
- Simple, fast to train, and works on large datasets.
- Highly **interpretable** — coefficients directly explain feature impact.
- A strong baseline before trying complex models.

**Cons**
- Assumes a linear relationship — underfits non-linear data.
- Sensitive to outliers (squared error magnifies them).
- Struggles with multicollinearity (correlated features → unstable coefficients).

**When to use**
- The relationship is roughly linear and you value interpretability.
- As a baseline to benchmark fancier models against.
- When you need to *explain* predictions, not just make them.

> If features are correlated or you need to prevent overfitting, move to **Ridge & Lasso** (Topic 2), which add regularisation to linear regression.

## 10. Summary

- Linear regression fits a line/hyperplane by minimising **Mean Squared Error**.
- It can be solved exactly (**normal equation**) or iteratively (**gradient descent**).
- On California Housing it explains roughly 60% of the variance in house value, with **median income** as the dominant driver.
- Our from-scratch gradient descent matched scikit-learn — proving we understand the mechanics, not just the API.